# Lecture 7: Switches as events: integrating discontinuous dynamics

**The challenge.** Lecture 4 showed that bounded controls are rarely smooth.
A bang-bang throttle jumps from off to full; a clipped force has corners
where it meets its bound. The canonical equations inherit those jumps: their
right-hand side $F$ is *discontinuous* at a switch and has a *kink* at a
saturation corner. Runge–Kutta methods (Lecture 6) assume $F$ is smooth
inside each step. A step that straddles a switch is wrong to first order in
the step size, whatever the method's order. The error estimate notices and
the step shrinks, again and again, until the integrator has crawled across
the jump with a cloud of tiny steps. The result is slow. Worse,
the switch time itself, which Lecture 8 must differentiate, is never
computed at all.

The cure is to treat each switch as an **event**: find the instant where the
switching function crosses zero, stop exactly there, change the control, and
restart. This lecture shows how the package does that, and what it costs
not to.

**After this lecture you can**

- describe an extremal as a hybrid system: modes, guards and resets;
- read switch times, mode flags and status codes from `propagate`;
- explain saturation kinks, tangential switches and the switch limit.

## 1. Equations with a broken right-hand side

Ordinary differential equation theory assumes a right-hand side continuous
in $y$. Bang-bang control broke that assumption, and the Soviet school that
produced the minimum principle also produced the theory for it. Aleksei
Fyodorovich Filippov at Moscow University defined in 1960 what a solution
of $\dot y = F(y)$ means when $F$ jumps across a surface. Where trajectories
cross the surface, the solution simply continues on the other side. Where
the vector fields on both sides push *into* the surface, the solution slides
*along* it, with a velocity that is a convex combination of the two. Vadim
Utkin later built a whole style of control on such *sliding modes*. In
optimal control the sliding case is the singular arc of Lecture 12. Here we
deal with the crossing case.

On the numerical side, root-finding inside a general-purpose ODE code became
standard in the early 1980s, with Linda Petzold and Alan Hindmarsh's LSODAR
at Lawrence Livermore. The idea is simple. Monitor a function $g(t, y)$
along the solution; when it changes sign within a step, locate the root on
the step's dense output, and stop there.

## 2. The extremal as a hybrid system

Write the control law of Lecture 4 for a bang-bang control as
$u = u_{\text{hi}}$ where $S < 0$ and $u_{\text{lo}}$ where $S > 0$. Between
switches, $F$ is smooth: it is $F$ with $u$ fixed at one bound. So the
extremal is a **hybrid system**:

- a **mode**, or **arc**: one flag per bang-bang control, saying which bound
  it holds;
- on each arc, smooth dynamics $\dot z = F_{\text{arc}}(t, z)$;
- a **guard** per control, its switching function $S(t, z)$, whose zero
  ends the arc;
- a **reset** at the guard: flip that control's flag. The state is
  continuous; the STM jumps (Lecture 8).

The package generates every piece. `switching` evaluates the guards,
`initial_arc` picks the flags from their signs at the start, and `rhs_arc`
evaluates $F$ on a given arc. Propagation is one FNI hybrid integration,
with each guard located to round-off on the dense output, so no step ever
straddles a switch, and the step size carries across.

Take the package's fuel-optimal low-thrust transfer between circular
orbits: a spacecraft that coasts and burns, with the throttle bang-bang on
the switching function of Lecture 4.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from auto_optimal_control import examples
from auto_optimal_control.build import build_kernels
from auto_optimal_control.numeric import NumericSystem
from auto_optimal_control.pontryagin import derive

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

pmp = derive(examples.planar_low_thrust_fuel())
k = build_kernels(pmp)
p = [1.0, 0.05, 1.0, 1.0, 1.0, 1.5, 6.0]            # mu, T_max, c, m0, r0, rf, tf
z0 = np.array([1.0, 0.0, 0.0, 1.0, 1.0, 0.2, 0.1, 0.1, 0.95, 0.0])
print("bang-bang controls:", k.N_SWITCHING, " mode flags:", k.N_MODES)
print("S at t = 0:", k.switching(0.0, z0, p), "-> initial arc", k.initial_arc(0.0, z0, p))

r = k.propagate(0.0, z0, 6.0, p, cost=True, rtol=1e-12, atol=1e-12)
print("status:", r["status"], " switches at t =", np.round(r["switch_times"], 8),
      " (control", r["switch_index"], ")")
print("arc at start and end:", r["arc0"], r["arc"], " propellant used:", round(r["cost"], 6))

Coast, burn, coast, burn: three switches, each located and reported, with
the index of the control that switched. The arc flags at the two ends say
which bound the throttle held.

In [ ]:
tr = k.trajectory(0.0, z0, 6.0, p, n_points=1201, rtol=1e-12, atol=1e-12)
S = np.array([k.switching(ti, zi, p)[0] for ti, zi in zip(tr["t"], tr["z"])])

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 3.4))
burn = tr["u"][:, 2] > 0.5
ax1.plot(tr["z"][~burn, 0], tr["z"][~burn, 1], ".", color=GRAY, ms=1.5, label="coast")
ax1.plot(tr["z"][burn, 0], tr["z"][burn, 1], ".", color=ORANGE, ms=2, label="burn")
th = np.linspace(0, 2 * np.pi, 200)
for rad in (1.0, 1.5):
    ax1.plot(rad * np.cos(th), rad * np.sin(th), color="k", lw=0.5, ls="--")
ax1.set_aspect("equal")
ax1.legend(frameon=False, fontsize=8, markerscale=6)
ax1.set_title("the path (dashed: start and target orbits)", fontsize=10, loc="left")
ax2.plot(tr["t"], S, color=BLUE, lw=2)
ax2.axhline(0, color="k", lw=0.8)
ax2.set_title("switching function S", fontsize=10, loc="left")
ax3.plot(tr["t"], tr["u"][:, 2], color=ORANGE, lw=2)
ax3.set_title(r"throttle $\delta$", fontsize=10, loc="left")
for ax in (ax2, ax3):
    for ts in r["switch_times"]:
        ax.axvline(ts, color=GRAY, lw=0.8, ls=":")
    ax.set_xlabel("t")
for ax in (ax1, ax2, ax3):
    ax.grid(alpha=0.3)
fig.tight_layout()

(These costates are a test point, not a solution: the spacecraft ends near
the target orbit, but not on it with the right velocity, so the terminal
conditions are not met. Lecture 10 solves the real transfer.)

## 3. What locating the switches buys

Compare with an integrator that does not know about the switches: the same
equations, evaluated from the NumPy reference with the bang-bang law as a
`Piecewise`, given to FNI's `solve_ivp` with no events. Both are DOP853. The
reference is `propagate` at a very tight tolerance.

In [ ]:
from fast_numerical_integration.integrators import solve_ivp

num = NumericSystem(pmp)
ref = k.propagate(0.0, z0, 6.0, p, rtol=1e-14, atol=1e-14)["z"]
rows = []
for rtol in [1e-6, 1e-8, 1e-10, 1e-12]:
    ev = k.propagate(0.0, z0, 6.0, p, rtol=rtol, atol=rtol * 1e-2)
    naive = solve_ivp(num.ode(p), (0.0, 6.0), z0, method="DOP853", rtol=rtol, atol=rtol * 1e-2)
    rows.append((rtol, np.abs(ev["z"] - ref).max(), ev["nfev"],
                 np.abs(naive.y[:, -1] - ref).max(), naive.nfev))
print(" rtol   | with events: error   RHS evals | no events: error   RHS evals")
for rtol, e1, n1, e2, n2 in rows:
    print(f" {rtol:.0e} |          {e1:8.1e}  {n1:8d}   |          {e2:8.1e}  {n2:8d}")

Both reach about the requested accuracy here, because DOP853's error control
is robust: at each jump it rejects step after step until the step is small
enough that the jump hardly matters. That is where the cost goes. Without
events the integration takes four to six times as many evaluations, nearly
all of them spent crawling across three switches. And it never learns *when*
the switches happened, which Lecture 8 needs: the STM jumps at each switch
by an amount that depends on the switch time.

## 4. Kinks: saturation corners

A clipped control (a quadratic $H$ with a box, Lecture 4) is continuous, so
$F$ does not jump. But $F$ has a *corner* wherever the unclipped law crosses
a bound, and a corner costs a high-order method most of its order. The
package treats these corners the same way: each is a guard (`kinks`), and
the integration restarts there. There is nothing to reset: the flow is
continuous and so is its STM.

In [ ]:
kb = build_kernels(derive(examples.double_integrator_min_energy(bounded=True)))
pb = [2.0, 1.0, 0.5, 0.8]                          # T, a, b, u_max
zb = np.array([1.0, 0.5, 1.5, 1.6])
print("kink guards:", kb.N_KINKS, " values at t = 0:", kb.kinks(0.0, zb, pb))
trb = kb.trajectory(0.0, zb, 2.0, pb, n_points=401)
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.plot(trb["t"], trb["u"][:, 0], color=BLUE, lw=2)
ax.axhline(-0.8, color="k", lw=0.8)
ax.axhline(0.8, color="k", lw=0.8)
ax.set(xlabel="t", ylabel="u = clip(-lam_v)")
ax.grid(alpha=0.3)
fig.tight_layout()

## 5. When switching goes wrong

Two situations need care, and `propagate` reports both in `status` (0 means
the integration finished):

- **Tangential switches** ($\dot S = 0$ at the zero). The switching function
  touches zero without crossing it. The switch time is then not a smooth
  function of the initial conditions, and the STM jump of Lecture 8 divides
  by $\dot S$. Propagation stops with status `-3`. In practice it happens at
  the entry of singular arcs (Lecture 12), which the package handles
  separately.
- **Too many switches.** A trajectory can chatter, switching faster and
  faster, which happens near singular arcs of order two (Fuller's
  phenomenon, Lecture 12). `max_switches` (default 10000) caps the count,
  and propagation stops with status `-2`.

Status `-1` means the step size underflowed: the trajectory ran into a
singularity, for example a spacecraft falling into the centre of attraction.
Status `-4` belongs to path constraints (Lecture 13). Shooting treats any
nonzero status as a failed evaluation.

In [ ]:
r_short = k.propagate(0.0, z0, 6.0, p, max_switches=2)
print("with max_switches=2: status", r_short["status"], "at t =", round(r_short["t"], 4),
      "after switches", np.round(r_short["switch_times"], 4))

## Summary

- Bang-bang and clipped controls make the canonical equations
  discontinuous or kinked; integrating across them loses accuracy and costs
  steps.
- The package integrates an extremal as a hybrid system: arc flags per
  bang-bang control, smooth dynamics on each arc (`rhs_arc`), switching
  functions as guards (`switching`, `initial_arc`), and a flip of the flag
  at each zero, located to round-off.
- Saturation corners are guards too (`kinks`), without a reset.
- `propagate` reports `switch_times`, `switch_index`, `arc0`, `arc` and a
  `status`: 0 done, -1 step too small, -2 too many switches, -3 tangential
  switch, -4 constraint violated.

## Exercises

1. Plot the step sizes of the naive integration of section 3 (the
   differences of `naive.t`) on a log scale, and mark the switch times. What
   happens near each switch?
2. Find costates for the low-thrust example that give no switch at all.
   (Hint: make $S$ positive at the start and keep its slope small.) How does
   the cost of `propagate` change?
3. For the minimum-time double integrator, $S = \lambda_v$ is linear in time.
   Which initial costates make the switch tangential? What does the
   trajectory look like there?

## Further reading

- A. F. Filippov, *Differential Equations with Discontinuous Righthand
  Sides* (Kluwer, 1988), the theory in full.
- L. F. Shampine and S. Thompson, "Event location for ordinary differential
  equations", *Computers & Mathematics with Applications* 39, 2000.
- E. Hairer, S. P. Nørsett and G. Wanner, *Solving Ordinary Differential
  Equations I*, section II.6, on dense output and discontinuities.